# 실습 14. 청구자료에서 코호트 만들기

사회약학 연구방법 노트의 '실습 14 청구자료에서 코호트 만들기'에 딸린 노트북입니다. 청구자료 모양의 가상 자료 세 표(자격, 명세서·상병, 처방)에서 SGLT2 억제제와 DPP-4 억제제 신규 사용자 코호트를 만들고, 추적 기간과 심부전 입원의 발생률, 위험비를 구합니다. 자료는 14장 예제 연구의 30분의 1 표본이며 실제 환자 자료가 아닙니다. 셀을 위에서부터 차례로 실행하세요. 각 셀의 설명과 출력 읽는 법은 사이트의 실습 14 쪽에 있습니다.


## 가. 청구자료의 표 구조

자격, 명세서·상병, 처방 세 표를 불러와 행의 단위를 확인하고 한 사람의 기록을 따라가 봅니다.

### 셀 1. lifelines 설치

In [ ]:
!pip install lifelines

### 셀 2. 세 표 불러오기

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

BASE = "https://socialp-ajou.tecentriq12.workers.dev/data/"
UA = {"User-Agent": "Mozilla/5.0"}   # 사이트가 파이썬 기본 요청을 막아 브라우저처럼 보이게 함
person = pd.read_csv(BASE + "claims_person.csv",
    parse_dates=["elig_start", "elig_end", "death_date"],
    storage_options=UA)
visit = pd.read_csv(BASE + "claims_visit.csv",
                    parse_dates=["visit_date"], storage_options=UA)
rx = pd.read_csv(BASE + "claims_rx.csv", parse_dates=["rx_date"],
                 storage_options=UA)
print(person.shape, visit.shape, rx.shape)
print(rx.dtypes)

### 셀 3. 표마다 앞의 세 줄 보기

In [ ]:
print(person.head(3))
print(visit.head(3))
print(rx.head(3))

### 셀 4. 행의 단위 확인하기

In [ ]:
print("person:", len(person), "rows,",
      person["pid"].nunique(), "people")
print("visit :", len(visit), "rows,",
      visit["pid"].nunique(), "people")
print("rx    :", len(rx), "rows,", rx["pid"].nunique(), "people")
print(person["elig_start"].min().date(),
      person["elig_end"].max().date())
print(rx.groupby("pid").size().describe().round(1))
print(rx["drug"].value_counts())
print(visit["setting"].value_counts())

### 셀 5. 한 사람의 기록을 세 표에서 따라가기

In [ ]:
p = 11891
print(person[person["pid"] == p])
print(rx[rx["pid"] == p].sort_values("rx_date"))
print(visit[visit["pid"] == p].sort_values("visit_date"))

## 나. 신규 사용자 코호트 만들기

index date를 찾고, 선정 기준을 하나씩 표지로 만든 뒤 차례로 적용하며 인원을 셉니다.

### 셀 6. 등록 기간의 첫 처방일(index date) 찾기

In [ ]:
study = rx[rx["drug"].isin(["SGLT2i", "DPP4i"])]
enroll = study[(study["rx_date"] >= "2016-01-01")
               & (study["rx_date"] <= "2021-12-31")]
index = enroll.groupby("pid")["rx_date"].min()
index = index.rename("index_date").reset_index()
print(study["pid"].nunique(), "->", len(index), "people")
index.head(3)

### 셀 7. index date에 처방받은 계열과 동시 시작 여부

In [ ]:
first = enroll.merge(index, on="pid")
first = first[first["rx_date"] == first["index_date"]]
print(len(enroll), "->", len(first), "rows")

coh = index.copy()
coh["arm"] = coh["pid"].map(first.groupby("pid")["drug"].first())
n_class = first.groupby("pid")["drug"].nunique()
coh["both"] = coh["pid"].map(n_class) > 1
print(coh["arm"].value_counts())
print("both classes on index date:", coh["both"].sum())

### 셀 8. 세척 기간 365일의 처방으로 기존 사용자 찾기

In [ ]:
m = study.merge(index, on="pid")
m["before"] = (m["index_date"] - m["rx_date"]).dt.days
prior = m[(m["before"] >= 1) & (m["before"] <= 365)]
coh["prevalent"] = coh["pid"].isin(prior["pid"])
print("prevalent users:", coh["prevalent"].sum())
m[m["pid"] == 10111]

### 셀 9. 자격 표를 붙여 index date의 나이 구하기

In [ ]:
info = person[["pid", "sex", "birth_year", "death_date"]]
coh = coh.merge(info, on="pid", how="left")
coh["age"] = coh["index_date"].dt.year - coh["birth_year"]
coh["minor"] = coh["age"] < 18
print(coh["age"].describe().round(1))
print("age < 18:", coh["minor"].sum())

### 셀 10. 기저 기간의 상병으로 말기신부전과 심부전 입원력 찾기

In [ ]:
v = visit.merge(index, on="pid")
v["before"] = (v["index_date"] - v["visit_date"]).dt.days
base = v[(v["before"] >= 0) & (v["before"] <= 365)]

def has_dx(df, codes, cols=("dx_main", "dx_sub1", "dx_sub2")):
    hit = pd.Series(False, index=df.index)
    for col in cols:
        hit = hit | df[col].str.startswith(codes, na=False)
    return hit

esrd = base[has_dx(base, ("N185", "Z49", "Z992"))]
hhf0 = base[(base["setting"] == "I")
            & has_dx(base, "I50", cols=["dx_main"])]
coh["esrd"] = coh["pid"].isin(esrd["pid"])
coh["prior_hhf"] = coh["pid"].isin(hhf0["pid"])
print(coh[["esrd", "prior_hhf"]].sum())

### 셀 11. 선정 기준을 차례로 적용하며 인원 세기

In [ ]:
steps = [("prevalent", "Prior use of either class (365 d)"),
         ("both", "Both classes on index date"),
         ("minor", "Age < 18 years"),
         ("esrd", "ESRD or dialysis (365 d)"),
         ("prior_hhf", "HF hospitalization (365 d)")]

def flow_table(coh):
    keep = pd.Series(True, index=coh.index)
    rows = [("Prescribed in 2016-2021", "", len(coh))]
    for col, label in steps:
        out = keep & coh[col]
        keep = keep & ~coh[col]
        rows.append((label, out.sum(), keep.sum()))
    flow = pd.DataFrame(rows,
                        columns=["step", "excluded", "remaining"])
    return flow, keep

flow, keep = flow_table(coh)
print(flow.to_string(index=False))
cohort = coh[keep].copy()
print(cohort["arm"].value_counts())

## 다. 노출·결과·공변량 만들기

처방 기록에서 중단일과 변경일을, 명세서에서 심부전 입원일과 기저 동반질환을 만듭니다.

### 셀 12. 처방 기록에서 약이 떨어지는 날과 공백 구하기

In [ ]:
f = study.merge(cohort[["pid", "index_date", "arm"]], on="pid")
own = f[(f["drug"] == f["arm"])
        & (f["rx_date"] >= f["index_date"])].copy()
own = own.sort_values(["pid", "rx_date"])
own["runout"] = own["rx_date"] + pd.to_timedelta(own["days"],
                                                 unit="D")
own["next"] = own.groupby("pid")["rx_date"].shift(-1)
own["gap"] = (own["next"] - own["runout"]).dt.days
own.loc[own["pid"] == p, ["rx_date", "days", "runout", "gap"]]

### 셀 13. 유예기간으로 중단일 정하기

In [ ]:
def stop_date(own, grace):
    over = own["gap"].isna() | (own["gap"] > grace)
    runout = own[over].groupby("pid")["runout"].first()
    return runout + pd.Timedelta(days=grace)

cohort["stop_date"] = cohort["pid"].map(stop_date(own, 60))
for grace in [30, 60, 90]:
    print("grace", grace, "->", stop_date(own, grace)[p].date())

### 셀 14. 비교약으로 바꾸거나 더한 날

In [ ]:
other = f[(f["drug"] != f["arm"])
          & (f["rx_date"] > f["index_date"])]
switch = other.groupby("pid")["rx_date"].min()
cohort["switch_date"] = cohort["pid"].map(switch)
print("switched or added:", cohort["switch_date"].notna().sum())
print(cohort["switch_date"].notna().groupby(cohort["arm"]).mean()
      .round(3))

### 셀 15. 결과: 주상병이 심부전인 입원의 첫 날짜

In [ ]:
after = visit.merge(cohort[["pid", "index_date"]], on="pid")
after = after[after["visit_date"] > after["index_date"]]
adm = after[(after["setting"] == "I")
            & has_dx(after, "I50", cols=["dx_main"])]
cohort["hhf_date"] = cohort["pid"].map(
    adm.groupby("pid")["visit_date"].min())
print("primary, inpatient :", len(adm), "claims,",
      adm["pid"].nunique(), "people")
wide = after[(after["setting"] == "I") & has_dx(after, "I50")]
print("any position, inpt :", wide["pid"].nunique(), "people")
print("any claim with I50 :",
      after[has_dx(after, "I50")]["pid"].nunique(), "people")

### 셀 16. 기저 기간의 상병과 처방으로 공변량 만들기

In [ ]:
b = base[base["pid"].isin(cohort["pid"])]
dx_def = {"htn": ("I10", "I11", "I12", "I13", "I15"),
          "ihd": ("I20", "I21", "I22", "I23", "I24", "I25"),
          "hf": "I50", "ckd": "N18"}
for name, codes in dx_def.items():
    pids = b[has_dx(b, codes)]["pid"]
    cohort[name] = cohort["pid"].isin(pids).astype(int)

r = rx.merge(index, on="pid")
r["before"] = (r["index_date"] - r["rx_date"]).dt.days
rb = r[(r["before"] >= 0) & (r["before"] <= 365)]
for name, drug in [("met", "metformin"), ("ins", "insulin")]:
    pids = rb[rb["drug"] == drug]["pid"]
    cohort[name] = cohort["pid"].isin(pids).astype(int)

cohort["female"] = (cohort["sex"] == "F").astype(int)
cohort["sglt2"] = (cohort["arm"] == "SGLT2i").astype(int)
covs = ["age", "female", "htn", "ihd", "hf", "ckd", "met", "ins"]
cohort[["pid", "arm", "index_date"] + covs].head(3)

## 라. 추적 기간과 발생률

추적 종료일을 정하고 인년, 발생률, 기저 특성 표, 누적발생 곡선, 위험비를 구합니다.

### 셀 17. 추적 종료일, 추적 일수, 사건 여부

In [ ]:
cohort["study_end"] = pd.Timestamp("2022-12-31")

def follow_up(c, ends):
    c = c.copy()
    c["end_date"] = c[ends].min(axis=1)
    c["time"] = (c["end_date"] - c["index_date"]).dt.days
    c["event"] = (c["hhf_date"] == c["end_date"]).astype(int)
    hit = [c[col] == c["end_date"] for col in ends]
    c["reason"] = np.select(hit, ends, default="")
    return c

AT = ["hhf_date", "death_date", "switch_date", "stop_date",
      "study_end"]
cohort = follow_up(cohort, AT)
print(cohort.loc[cohort["pid"] == p,
                 ["index_date", "end_date", "time", "event"]])
pd.crosstab(cohort["reason"], cohort["arm"], margins=True)

### 셀 18. 사건 수, 인년, 1,000인년당 발생률과 발생률비

In [ ]:
from scipy import stats

def rate_table(c):
    t = c.groupby("arm").agg(n=("event", "size"),
                             events=("event", "sum"),
                             py=("time", "sum"))
    t["py"] = t["py"] / 365.25
    d = t["events"]
    t["rate"] = 1000 * d / t["py"]
    t["lo"] = 1000 * stats.chi2.ppf(0.025, 2 * d) / 2 / t["py"]
    t["hi"] = 1000 * stats.chi2.ppf(0.975, 2 * d + 2) / 2 / t["py"]
    return t

def irr(t):
    ratio = t.loc["SGLT2i", "rate"] / t.loc["DPP4i", "rate"]
    se = np.sqrt((1 / t["events"]).sum())
    lo, hi = np.exp(-1.96 * se), np.exp(1.96 * se)
    return ratio, ratio * lo, ratio * hi

tab = rate_table(cohort)
print(tab.round(2))
print("IRR %.2f (%.2f-%.2f)" % irr(tab))

### 셀 19. 기저 특성 표

In [ ]:
g = cohort.groupby("arm")
t1 = g[covs].mean().T
t1.loc[covs[1:]] = t1.loc[covs[1:]] * 100      # 0/1 변수는 %
t1.loc["age SD"] = g["age"].std()
t1.loc["n"] = g.size()
t1.round(1)

### 셀 20. Kaplan-Meier 누적발생 곡선

In [ ]:
from lifelines import KaplanMeierFitter

fig, ax = plt.subplots(figsize=(6.5, 4))
for arm in ["DPP4i", "SGLT2i"]:
    s = cohort[cohort["arm"] == arm]
    km = KaplanMeierFitter()
    km.fit(s["time"] / 365.25, s["event"], label=arm)
    km.plot_cumulative_density(ax=ax)
ax.set_xlabel("Years since index date (as-treated)")
ax.set_ylabel("Cumulative incidence of HF hospitalization")
plt.tight_layout()

### 셀 21. 보정 전과 보정 후의 Cox 위험비

In [ ]:
from lifelines import CoxPHFitter

def hr(c, cols):
    m = CoxPHFitter().fit(c[["time", "event"] + cols],
                          "time", "event")
    s = m.summary.loc["sglt2"]
    return (s["exp(coef)"], s["exp(coef) lower 95%"],
            s["exp(coef) upper 95%"])

print("crude HR    %.2f (%.2f-%.2f)" % hr(cohort, ["sglt2"]))
print("adjusted HR %.2f (%.2f-%.2f)" % hr(cohort, ["sglt2"] + covs))

### 셀 22. 사이트의 분석용 코호트 파일과 대조하기

In [ ]:
ref = pd.read_csv(BASE + "sglt2_cohort.csv", storage_options=UA)
mine = cohort.sort_values("pid").reset_index(drop=True)
cols = ["pid", "sglt2", "time", "event"] + covs
print(len(mine), len(ref))
print((mine[cols] == ref[cols]).all().all())

### 셀 23. 전체 코호트 166,548명으로 같은 분석 하기

In [ ]:
full = pd.read_csv(BASE + "sglt2_cohort_full.csv", storage_options=UA)
full["arm"] = full["sglt2"].map({1: "SGLT2i", 0: "DPP4i"})

def summary(c):
    t = rate_table(c)
    out = {}
    for arm in ["SGLT2i", "DPP4i"]:
        r = t.loc[arm]
        out[arm + " events / PY"] = "%d / %.0f" % (r["events"],
                                                   r["py"])
        out[arm + " rate (95% CI)"] = "%.2f (%.2f-%.2f)" % (
            r["rate"], r["lo"], r["hi"])
    out["IRR (95% CI)"] = "%.2f (%.2f-%.2f)" % irr(t)
    out["crude HR (95% CI)"] = "%.2f (%.2f-%.2f)" % hr(c, ["sglt2"])
    out["adjusted HR (95% CI)"] = "%.2f (%.2f-%.2f)" % hr(
        c, ["sglt2"] + covs)
    return out

pd.DataFrame({"sample (n = 5,552)": summary(cohort),
              "full (n = 166,548)": summary(full)})

## 과제 정답

**과제 1.** 유예기간을 60일에서 30일로 바꿔 as-treated 추적을 다시 만들고, 군별 사건 수, 인년, 1,000인년당 발생률, 발생률비, 보정 위험비를 구합니다. 가 절에서 따라가 본 환자의 추적이 어떻게 달라지는지도 확인합니다.

### 셀 24. 과제 1 정답: 유예기간 30일

In [ ]:
c30 = cohort.copy()
c30["stop_date"] = c30["pid"].map(stop_date(own, 30))
c30 = follow_up(c30, AT)
t30 = rate_table(c30)
print(t30.round(2))
print("IRR %.2f (%.2f-%.2f)" % irr(t30))
print("adjusted HR %.2f (%.2f-%.2f)" % hr(c30, ["sglt2"] + covs))
print(c30.loc[c30["pid"] == p, ["end_date", "time", "event"]])

**과제 2.** 세척 기간을 365일에서 180일로 줄여 선정 흐름을 다시 만들고, 기존 사용자로 빠지는 인원과 최종 코호트 인원이 얼마나 달라지는지 구합니다.

### 셀 25. 과제 2 정답: 세척 기간 180일

In [ ]:
coh180 = coh.copy()
prior180 = m[(m["before"] >= 1) & (m["before"] <= 180)]
coh180["prevalent"] = coh180["pid"].isin(prior180["pid"])
flow180, keep180 = flow_table(coh180)
print(flow180.to_string(index=False))
print(coh180[keep180]["arm"].value_counts())
late = prior[~prior["pid"].isin(prior180["pid"])]
print(late.groupby("pid")["before"].min().describe().round(0))

**과제 3.** 약 중단과 비교약으로의 변경을 무시하는 ITT 방식으로 추적을 다시 만들고, 발생률, 발생률비, 보정 전과 보정 후의 위험비를 구합니다. 약을 끊거나 바꾼 뒤의 기간에서 발생률이 얼마인지도 계산하고, 전체 코호트 파일로 같은 분석을 해 본문 표 14-5의 ITT 유사 분석과 맞는지 확인합니다.

### 셀 26. 과제 3 정답: ITT 방식의 추적

In [ ]:
itt = follow_up(cohort, ["hhf_date", "death_date", "study_end"])
ti = rate_table(itt)
print(ti.round(2))
print("IRR %.2f (%.2f-%.2f)" % irr(ti))
print("crude HR    %.2f (%.2f-%.2f)" % hr(itt, ["sglt2"]))
print("adjusted HR %.2f (%.2f-%.2f)" % hr(itt, ["sglt2"] + covs))
off = ti[["events", "py"]] - tab[["events", "py"]]
print((1000 * off["events"] / off["py"]).round(2))

full_itt = full.copy()         # 전체 코호트의 ITT 추적
full_itt["time"] = full["time_itt"].fillna(full["time"])
full_itt["event"] = full["event_itt"].fillna(full["event"])
pd.DataFrame({"sample (n = 5,552)": summary(itt),
              "full (n = 166,548)": summary(full_itt)})